In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install ultralytics -q
import os, glob, zipfile, random, shutil, yaml
from ultralytics import YOLO
from IPython.display import FileLink

print("📦 Checking and preparing dataset...")

# 1. Automatic Dataset Location Finder
input_dir = '/kaggle/input'
extract_path = '/kaggle/working/dataset_raw'

zip_files = glob.glob(f"{input_dir}/**/*.zip", recursive=True)

if zip_files:
    print(f"📦 Extracting zip dataset: {zip_files[0]}")
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_files[0], 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    search_base = extract_path
else:
    search_base = input_dir

# Find images and labels directories dynamically
images_dirs = glob.glob(f"{search_base}/**/images", recursive=True)
labels_dirs = glob.glob(f"{search_base}/**/labels", recursive=True)

images_dir = images_dirs[0]
labels_dir = labels_dirs[0]

# 2. Train-Val Split Setup (80-20 Split)
final_dataset_path = '/kaggle/working/dataset'
for folder in ['train/images', 'train/labels', 'val/images', 'val/labels']:
    os.makedirs(os.path.join(final_dataset_path, folder), exist_ok=True)

all_images = [f for f in os.listdir(images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
random.seed(42)
random.shuffle(all_images)

split_index = int(len(all_images) * 0.8)
train_images, val_images = all_images[:split_index], all_images[split_index:]

def move_files(files, split_name):
    for filename in files:
        shutil.copy(os.path.join(images_dir, filename), os.path.join(final_dataset_path, split_name, 'images', filename))
        label_filename = os.path.splitext(filename)[0] + '.txt'
        if os.path.exists(os.path.join(labels_dir, label_filename)):
            shutil.copy(os.path.join(labels_dir, label_filename), os.path.join(final_dataset_path, split_name, 'labels', label_filename))

move_files(train_images, 'train')
move_files(val_images, 'val')

# 3. Create CORRECTED data.yaml
yaml_content = {
    'path': '/kaggle/working/dataset',
    'train': 'train/images',
    'val': 'val/images',
    'nc': 4,
    'names': ['debris', 'landslide', 'structures', 'uprooted_tree']
}
with open('/kaggle/working/dataset/data.yaml', 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print("✅ dataset & data.yaml ready!")
print("🚀 Starting YOLOv8s Training (30 Epochs)...")

# 4. Start YOLOv8s Model Training
model_v8 = YOLO("yolov8s.pt")
results_v8 = model_v8.train(
    data="/kaggle/working/dataset/data.yaml", 
    epochs=30,                          
    imgsz=640,                          
    batch=32,          
    device=0,          # GPU T4
    workers=4,
    name="yolov8s_results"
)

print("🎉 YOLOv8s Training Completed Successfully!")

# 5. Zip & Provide Download Link
shutil.make_archive('/kaggle/working/yolov8s_disaster_results', 'zip', '/kaggle/working/runs/detect/yolov8s_results')

print("✅ Zip created successfully! Download from below link:")
FileLink(r'yolov8s_disaster_results.zip')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.5 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
📦 Checking and preparing dataset...
✅ dataset & data.yaml ready!
🚀 Starting YOLOv8s Training (30 Epochs)...
Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dat

/kaggle/working/yolov8s_disaster_results.zip